# Entrega 05 — baselines, primeiro modelo de regressão e comparação

**Integrante:** Maria Dionila (`mariadionila`) — parte *Baselines, regressão e comparação* da equipe 02.
**Dados:** inteiramente **sintéticos** (mock). Nenhum número deste notebook descreve o RU real da UFPB.

## Formulação (herdada da E04, congelada)
Na véspera (`D-1`), prever a quantidade de refeições **servidas** em `D`, por `data × tipo_refeicao × curso`, para apoiar o planejamento da produção do RU. O total do RU é a soma dos cursos.

## O que este notebook faz
1. Reconstrói a base por curso a partir dos CSVs brutos (mesma regra da E04; não é tarefa nova).
2. Reutiliza os dois baselines da E04: **trivial** (média do treino) e **operacional proposto** (`D-7`).
3. Estima o **primeiro modelo de regressão**: Ridge dentro de `Pipeline` + `ColumnTransformer`, com validação cruzada temporal.
4. Compara os três **na mesma partição, com a mesma métrica e os mesmos casos**.
5. Examina os resíduos e registra limites.

**Partição (igual à E04):** treino 2019–2024, validação 2025, teste reservado 2026. O teste só é aberto uma vez, no fim, com o modelo já congelado.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEMENTE = 20261001  # mesma semente do gerador; o Ridge é determinístico, mas a fixamos por registro
np.random.seed(SEMENTE)

RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dados").is_dir()), None)
if RAIZ is None:
    raise FileNotFoundError("Abra o notebook a partir da pasta do projeto.")
BRUTOS = RAIZ / "dados/bruto/mock"
PROCESSADOS = RAIZ / "dados/processado"
FIGURAS = RAIZ / "output/figuras"
PROCESSADOS.mkdir(parents=True, exist_ok=True)
FIGURAS.mkdir(parents=True, exist_ok=True)

FIM_TREINO = pd.Timestamp("2024-12-31")
FIM_VALIDACAO = pd.Timestamp("2025-12-31")
CHAVE = ["data", "tipo_refeicao", "curso"]

## 1. Base por curso (reprodução da E04)
Contam-se apenas eventos `servida`. A grade `data × refeição × curso` é completa porque, no mock, a cobertura é conhecida; domingos são fechados. Em dados reais isso exigiria o arquivo de cobertura descrito na E04.

In [2]:
catraca = pd.read_csv(BRUTOS / "catraca_ru.csv", dtype="string")
estudantes = pd.read_csv(BRUTOS / "estudantes.csv", dtype="string")
catraca["data"] = pd.to_datetime(catraca["data_hora_refeicao"]).dt.normalize()
servidas = catraca.loc[catraca.situacao_evento.eq("servida")]

datas = pd.date_range("2019-01-01", "2026-11-14", freq="D")
refeicoes = sorted(catraca.tipo_refeicao.unique())
cursos = sorted(estudantes.curso.dropna().unique())
base = pd.MultiIndex.from_product([datas, refeicoes, cursos], names=CHAVE).to_frame(index=False)
base["aberto"] = base.data.dt.day_of_week.lt(6)
cont = servidas.groupby(CHAVE).id_registro_utilizacao.nunique().rename("quantidade_servida").reset_index()
base = base.merge(cont, how="left", on=CHAVE, validate="one_to_one")
base["quantidade_servida"] = base.quantidade_servida.fillna(0).astype(int)
assert base.quantidade_servida.sum() == len(servidas) == 221_600
assert len(base) == len(datas) * len(refeicoes) * len(cursos)
print("Linhas da base:", len(base), "| refeições servidas:", len(servidas))

Linhas da base: 51750 | refeições servidas: 221600


## 2. Variáveis disponíveis na véspera e partição temporal

Cada preditora precisa existir ao fim de `D-1`:

| Variável | Tipo | Por que pode ser usada |
|---|---|---|
| `lag_7` (quantidade do mesmo curso/refeição em `D-7`) | numérica | já ocorreu e a catraca fecha antes de `D-1` |
| `media_4sem` (média de `D-7, D-14, D-21, D-28`) | numérica | idem; suaviza o ruído de uma única semana |
| `tendencia` (dias desde 01/01/2019) | numérica | o calendário é conhecido; captura o crescimento da demanda |
| `dia_semana`, `mes` | categóricas | calendário conhecido; efeito **não linear** (domingo fechado, sazonalidade), por isso one-hot |
| `tipo_refeicao`, `curso` | categóricas | identificam a unidade prevista |

**Fora do modelo:** quantidade, tentativas negadas e eventos de `D`; sobras e quantidade preparada; faltas e notas finais do período. Os atributos defasados são calculados por **data** dentro de cada curso/refeição, nunca olhando para o futuro.

In [3]:
base = base.sort_values(["curso", "tipo_refeicao", "data"]).reset_index(drop=True)
g = base.groupby(["curso", "tipo_refeicao"]).quantidade_servida
for k in (7, 14, 21, 28):
    base[f"lag_{k}"] = g.shift(k)               # grade diária completa: shift(k) = exatamente k dias antes
base["media_4sem"] = base[["lag_7", "lag_14", "lag_21", "lag_28"]].mean(axis=1)
base["tendencia"] = (base.data - pd.Timestamp("2019-01-01")).dt.days
base["dia_semana"] = base.data.dt.day_of_week.astype(str)
base["mes"] = base.data.dt.month.astype(str)

modelavel = base.loc[base.aberto].sort_values(["data", "curso", "tipo_refeicao"]).reset_index(drop=True)
treino = modelavel.loc[modelavel.data.le(FIM_TREINO)].copy()
validacao = modelavel.loc[modelavel.data.gt(FIM_TREINO) & modelavel.data.le(FIM_VALIDACAO)].copy()
teste = modelavel.loc[modelavel.data.gt(FIM_VALIDACAO)].copy()
assert treino.data.max() < validacao.data.min() < validacao.data.max() < teste.data.min()
print("Linhas treino / validação / teste:", len(treino), len(validacao), len(teste))   # E04: 33822 / 5634 / 4914
print("Linhas sem lag_7 (apenas início de 2019):", int(treino.lag_7.isna().sum()), "no treino;",
      int(validacao.lag_7.isna().sum()), "na validação;", int(teste.lag_7.isna().sum()), "no teste")

Linhas treino / validação / teste: 33822 5634 4914
Linhas sem lag_7 (apenas início de 2019): 108 no treino; 0 na validação; 0 no teste


## 3. Baselines (reaproveitados da E04)

- **Trivial:** média da quantidade por `curso × refeição` aprendida **só no treino**.
- **Operacional proposto:** `D-7` (mesmo curso/refeição sete dias antes). É a regra que a equipe *propôs* para o piloto; a regra real do RU ainda não foi confirmada.

In [4]:
media_treino = (treino.groupby(["curso", "tipo_refeicao"]).quantidade_servida.mean()
                .rename("prev_trivial").reset_index())

def com_baselines(df):
    df = df.merge(media_treino, how="left", on=["curso", "tipo_refeicao"], validate="many_to_one")
    df["prev_operacional"] = df["lag_7"]
    return df

validacao = com_baselines(validacao)
teste = com_baselines(teste)
print("Validação sem baseline trivial:", int(validacao.prev_trivial.isna().sum()), "| sem D-7:", int(validacao.prev_operacional.isna().sum()))

Validação sem baseline trivial: 0 | sem D-7: 0


## 4. Primeiro modelo: regressão Ridge em `Pipeline`

**Especificação.** $\hat y_i = \beta_0 + \mathbf{x}_i^\top\boldsymbol\beta$, estimada por mínimos quadrados com penalidade $\ell_2$: $\min_\beta \sum_i (y_i-\hat y_i)^2+\alpha\lVert\beta\rVert_2^2$.

**Hipóteses assumidas.** (i) relação aproximadamente **linear** entre o alvo e `lag_7`, `media_4sem` e `tendencia`, com efeitos aditivos de calendário/curso/refeição; (ii) o padrão aprendido em 2019–2024 vale em 2025–2026 (mecanismo estável, exceto pela tendência linear); (iii) erros com variância aproximadamente constante — **suposição fraca**, pois a contagem tem variância crescente com a média; (iv) preditoras colineares (`lag_7` e `media_4sem`) tornam os coeficientes individuais instáveis, o que a penalidade $\ell_2$ amortece mas não elimina.

**Estimação.** A penalidade $\alpha$ é escolhida por `GridSearchCV` com `TimeSeriesSplit` (5 blocos: treina no passado e valida no bloco seguinte) **apenas dentro do treino 2019–2024**, com o MAE como critério. Imputação (mediana), escala e one-hot ficam dentro do `Pipeline`: a cada ajuste, aprendem só com o bloco de treino daquela rodada. Previsões negativas são truncadas em zero, pois quantidade não é negativa.

In [5]:
NUMERICAS = ["lag_7", "media_4sem", "tendencia"]
CATEGORICAS = ["dia_semana", "mes", "tipo_refeicao", "curso"]
FEATURES = NUMERICAS + CATEGORICAS

prep = ColumnTransformer([
    ("num", Pipeline([("faltantes", SimpleImputer(strategy="median")), ("escala", StandardScaler())]), NUMERICAS),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAS),
])
modelo = Pipeline([("prep", prep), ("reg", Ridge())])

cv = TimeSeriesSplit(n_splits=5)
busca = GridSearchCV(modelo, {"reg__alpha": [0.01, 0.1, 1, 10, 100, 1000]}, cv=cv,
                     scoring="neg_mean_absolute_error", refit=True)
busca.fit(treino[FEATURES], treino.quantidade_servida)    # treino ordenado por data: os blocos respeitam o tempo
cv_alfa = pd.DataFrame({"alpha": busca.cv_results_["param_reg__alpha"].astype(float),
                        "MAE_cv": -busca.cv_results_["mean_test_score"], "dp": busca.cv_results_["std_test_score"]})
display(cv_alfa.round(4))
modelo_final = busca.best_estimator_      # ajustado somente com o treino completo
print("alpha escolhido:", busca.best_params_["reg__alpha"])

num = modelo_final.named_steps["prep"].named_transformers_["num"]
print("Atributos após o pré-processamento:", len(modelo_final.named_steps["prep"].get_feature_names_out()))
print("Medianas aprendidas no treino:", dict(zip(NUMERICAS, num.named_steps["faltantes"].statistics_.round(2))))
print("Média e desvio aprendidos no treino (escala):", dict(zip(NUMERICAS, zip(num.named_steps["escala"].mean_.round(2), num.named_steps["escala"].scale_.round(2)))))
nomes = modelo_final.named_steps["prep"].get_feature_names_out()
coef = pd.Series(modelo_final.named_steps["reg"].coef_, index=nomes).sort_values(key=abs, ascending=False)
print("Coeficientes de maior magnitude (numéricas em desvios-padrão):")
display(coef.head(10).round(3).to_frame("coeficiente"))

,alpha,MAE_cv,dp
0,0.01,2.0464,0.7830
1,0.10,2.0353,0.7607
2,1.00,1.9437,0.5777
3,10.00,1.7015,0.0960
4,100.00,1.6597,0.0237
5,1000.00,1.7222,0.0553


alpha escolhido: 100
Atributos após o pré-processamento: 30
Medianas aprendidas no treino: {'lag_7': np.float64(4.0), 'media_4sem': np.float64(4.25), 'tendencia': np.float64(1095.0)}
Média e desvio aprendidos no treino (escala): {'lag_7': (np.float64(4.89), np.float64(3.64)), 'media_4sem': (np.float64(4.88), np.float64(3.14)), 'tendencia': (np.float64(1095.25), np.float64(632.82))}
Coeficientes de maior magnitude (numéricas em desvios-padrão):


,coeficiente
cat__mes_7,-2.500
cat__mes_1,-2.079
cat__dia_semana_5,-1.879
cat__tipo_refeicao_almoco,1.442
cat__mes_2,1.149
cat__mes_8,1.116
cat__tipo_refeicao_jantar,-0.830
cat__mes_12,-0.782
num__media_4sem,0.753
cat__curso_CURSO_TECNICO_D,0.686


## 5. Validação cruzada temporal: baselines e modelo nos mesmos blocos

Para uma comparação justa dentro do treino, os dois baselines e o modelo são avaliados **nos mesmos cinco blocos temporais** e nos mesmos casos (os que têm `D-7`). A média trivial é recalculada em cada bloco só com o passado dele. O `alpha` já foi escolhido nesses mesmos blocos, então o MAE de CV do modelo é levemente otimista; por isso a comparação final usa a validação de 2025, não esta tabela.

In [6]:
linhas_cv = []
for i, (ia, ib) in enumerate(cv.split(treino), start=1):
    a, b = treino.iloc[ia], treino.iloc[ib]
    m = a.groupby(["curso", "tipo_refeicao"]).quantidade_servida.mean().rename("prev_trivial").reset_index()
    b = b.merge(m, how="left", on=["curso", "tipo_refeicao"])
    mod = clone(modelo_final).fit(a[FEATURES], a.quantidade_servida)
    b["prev_modelo"] = np.maximum(mod.predict(b[FEATURES]), 0)
    b = b.dropna(subset=["lag_7"])
    for nome, col in [("Baseline trivial", "prev_trivial"), ("Baseline operacional (D-7)", "lag_7"), ("Modelo Ridge", "prev_modelo")]:
        linhas_cv.append({"bloco": i, "abordagem": nome, "MAE": (b.quantidade_servida - b[col]).abs().mean()})
cv_tab = pd.DataFrame(linhas_cv).pivot(index="bloco", columns="abordagem", values="MAE")
display(cv_tab.round(3))
display(cv_tab.mean().round(3).rename("MAE médio nos 5 blocos").to_frame())

abordagem,Baseline operacional (D-7),Baseline trivial,Modelo Ridge
bloco,,,
1,2.107,2.486,1.664
2,2.058,2.460,1.615
3,2.111,2.472,1.655
4,2.047,2.433,1.612
5,2.028,2.411,1.596


,MAE médio nos 5 blocos
abordagem,
Baseline operacional (D-7),2.070
Baseline trivial,2.453
Modelo Ridge,1.628


## 6. Métricas e quadro obrigatório de comparação

**Métrica principal: MAE em refeições por `dia × refeição × curso`.** Justificativa pela decisão: o RU prepara refeições inteiras; subestimar deixa estudantes sem refeição (custo de falta, **B**) e superestimar gera sobra (custo de sobra, **C**). Como `B` e `C` ainda **não foram levantados** (nem a capacidade `k`), não há base para uma perda assimétrica; o MAE trata os dois erros com o mesmo peso e está na unidade da decisão (refeições). Se `B ≠ C`, a métrica deverá ser trocada por uma perda ponderada (por exemplo, quantílica).

**Métricas secundárias:** RMSE (penaliza erros grandes), **viés médio** `real − previsto` (positivo = subestima, sinal de falta) e MAE do **total diário do RU**. MAPE não é usado: com demanda por curso próxima de zero, ele explode.

In [7]:
def metricas(df, col):
    e = df.quantidade_servida - df[col]
    tot = df.groupby("data")[["quantidade_servida", col]].sum()
    return {"MAE (por linha)": e.abs().mean(), "RMSE (por linha)": np.sqrt((e ** 2).mean()),
            "Viés (real−prev.)": e.mean(), "MAE total diário": (tot.quantidade_servida - tot[col]).abs().mean(),
            "n linhas": len(df)}

ROTULOS = {"prev_trivial": "Baseline trivial (média do treino por curso×refeição)",
           "prev_operacional": "Baseline operacional proposto (D-7)",
           "prev_modelo": "Modelo Ridge (pipeline)"}

def avaliar(df):
    df = df.copy()
    df["prev_modelo"] = np.maximum(modelo_final.predict(df[FEATURES]), 0)
    assert df[list(ROTULOS)].notna().all().all()      # mesmos casos para os três
    return df, pd.DataFrame({ROTULOS[c]: metricas(df, c) for c in ROTULOS}).T

validacao, quadro_val = avaliar(validacao)
print("VALIDAÇÃO 2025 — mesma partição, mesmos", len(validacao), "casos, mesma métrica:")
display(quadro_val.round(3))

VALIDAÇÃO 2025 — mesma partição, mesmos 5634 casos, mesma métrica:


,MAE (por linha),RMSE (por linha),Viés (real−prev.),MAE total diário,n linhas
Baseline trivial (média do treino por curso×refeição),2.667,3.238,0.279,42.907,5634.0
Baseline operacional proposto (D-7),2.139,2.951,0.007,15.045,5634.0
Modelo Ridge (pipeline),1.703,2.238,0.194,14.217,5634.0


In [8]:
# MAE por tipo de refeição (mesmos casos)
por_ref = pd.DataFrame({ROTULOS[c]: validacao.assign(e=(validacao.quantidade_servida - validacao[c]).abs()).groupby("tipo_refeicao").e.mean()
                        for c in ROTULOS})
display(por_ref.round(3))

# Diferença pareada do erro absoluto, modelo vs D-7: o ganho é distinguível de ruído?
d = (validacao.quantidade_servida - validacao.prev_operacional).abs() - (validacao.quantidade_servida - validacao.prev_modelo).abs()
rng = np.random.default_rng(SEMENTE)
dias = d.groupby(validacao.data).mean().to_numpy()     # blocos por dia: erros do mesmo dia são correlacionados
boot = np.array([rng.choice(dias, len(dias), replace=True).mean() for _ in range(5000)])
print(f"Ganho médio de MAE do modelo sobre D-7: {d.mean():.3f} refeições/linha | IC 95% (bootstrap por dia): [{np.percentile(boot,2.5):.3f}; {np.percentile(boot,97.5):.3f}]")
print(f"Modelo melhor que D-7 em {(d>0).mean():.1%} das linhas, pior em {(d<0).mean():.1%} (empates: {(d==0).mean():.1%})")

,Baseline trivial (média do treino por curso×refeição),Baseline operacional proposto (D-7),Modelo Ridge (pipeline)
tipo_refeicao,,,
almoco,3.979,2.557,2.254
cafe_da_manha,2.136,2.008,1.459
jantar,1.887,1.853,1.396


Ganho médio de MAE do modelo sobre D-7: 0.437 refeições/linha | IC 95% (bootstrap por dia): [0.357; 0.521]
Modelo melhor que D-7 em 55.8% das linhas, pior em 41.1% (empates: 3.1%)


## 7. Teste reservado (2026) — abertura única, modelo congelado

Especificação e hiperparâmetros foram definidos só com treino e validação. O teste é calculado **uma vez**, com o mesmo modelo (ajustado no treino 2019–2024) e os mesmos baselines, para verificar se o resultado de 2025 se mantém.

In [9]:
teste, quadro_teste = avaliar(teste)
print("TESTE 2026 (até 14/11) — mesmos", len(teste), "casos:")
display(quadro_teste.round(3))
saida = pd.concat([quadro_val.assign(particao="validacao_2025"), quadro_teste.assign(particao="teste_2026")]).rename_axis("abordagem").reset_index()
saida.to_csv(PROCESSADOS / "resultados_comparacao_e05.csv", index=False)
cv_tab.to_csv(PROCESSADOS / "resultados_cv_temporal_e05.csv")

TESTE 2026 (até 14/11) — mesmos 4914 casos:


,MAE (por linha),RMSE (por linha),Viés (real−prev.),MAE total diário,n linhas
Baseline trivial (média do treino por curso×refeição),2.882,3.486,0.693,48.292,4914.0
Baseline operacional proposto (D-7),2.198,3.011,0.103,13.839,4914.0
Modelo Ridge (pipeline),1.773,2.324,0.453,15.381,4914.0


## 8. Resíduos (validação 2025)
Resíduo = real − previsto. Positivo significa **subestimativa** (risco de falta); negativo, superestimativa (sobra). Os gráficos são salvos em `output/figuras/` para o relatório.

In [10]:
v = validacao.copy()
v["res_modelo"] = v.quantidade_servida - v.prev_modelo
v["res_d7"] = v.quantidade_servida - v.prev_operacional
v["res_trivial"] = v.quantidade_servida - v.prev_trivial

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
bins = np.arange(-15.5, 16.5, 1)
for c, r, cor in [("Trivial", "res_trivial", "#999999"), ("D-7", "res_d7", "#d95f02"), ("Modelo Ridge", "res_modelo", "#1b6ca8")]:
    ax[0].hist(v[r], bins=bins, alpha=0.55, label=c, color=cor)
ax[0].set(title="Distribuição dos resíduos (2025)", xlabel="real − previsto (refeições)", ylabel="linhas"); ax[0].legend()
ax[1].scatter(v.prev_modelo, v.res_modelo, s=4, alpha=0.3, color="#1b6ca8"); ax[1].axhline(0, color="k", lw=0.8)
ax[1].set(title="Resíduo × previsto (modelo)", xlabel="previsto", ylabel="resíduo")
mensal = v.assign(m=v.data.dt.to_period("M").dt.to_timestamp()).groupby("m")[["res_trivial", "res_d7", "res_modelo"]].mean()
for c, cor in [("res_trivial", "#999999"), ("res_d7", "#d95f02"), ("res_modelo", "#1b6ca8")]:
    ax[2].plot(mensal.index, mensal[c], marker="o", label=c.replace("res_", ""), color=cor)
ax[2].axhline(0, color="k", lw=0.8); ax[2].set(title="Viés médio mensal (2025)", ylabel="real − previsto"); ax[2].tick_params(axis="x", rotation=45); ax[2].legend()
fig.tight_layout(); fig.savefig(FIGURAS / "e05_residuos_validacao.png", dpi=130); plt.close(fig)

nome_dia = {"0": "seg", "1": "ter", "2": "qua", "3": "qui", "4": "sex", "5": "sáb"}
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
por_dia = v.assign(dia=v.dia_semana.map(nome_dia), a_d7=v.res_d7.abs(), a_mod=v.res_modelo.abs()).groupby("dia")[["a_d7", "a_mod"]].mean().reindex(list(nome_dia.values()))
por_dia.columns = ["D-7", "Modelo Ridge"]
por_dia.plot.bar(ax=ax[0], color=["#d95f02", "#1b6ca8"], rot=0); ax[0].set(title="MAE por dia da semana (2025)", ylabel="refeições", xlabel="")
por_ref_g = v.assign(a_tri=v.res_trivial.abs(), a_d7=v.res_d7.abs(), a_mod=v.res_modelo.abs()).groupby("tipo_refeicao")[["a_tri", "a_d7", "a_mod"]].mean()
por_ref_g.columns = ["Trivial", "D-7", "Modelo Ridge"]
por_ref_g.plot.bar(ax=ax[1], color=["#999999", "#d95f02", "#1b6ca8"], rot=0); ax[1].set(title="MAE por refeição (2025)", ylabel="refeições", xlabel="")
fig.tight_layout(); fig.savefig(FIGURAS / "e05_mae_dia_refeicao.png", dpi=130); plt.close(fig)

fig, ax = plt.subplots(figsize=(11, 3.8))
tot = v.groupby("data")[["quantidade_servida", "prev_operacional", "prev_modelo"]].sum()
ax.plot(tot.index, tot.quantidade_servida, color="k", lw=1, label="real"); ax.plot(tot.index, tot.prev_operacional, color="#d95f02", lw=0.8, alpha=0.8, label="D-7")
ax.plot(tot.index, tot.prev_modelo, color="#1b6ca8", lw=0.8, alpha=0.9, label="Ridge"); ax.legend(); ax.set(title="Total diário do RU em 2025 (dias abertos)", ylabel="refeições")
fig.tight_layout(); fig.savefig(FIGURAS / "e05_total_diario.png", dpi=130); plt.close(fig)

print("Resumo dos resíduos do modelo (2025):"); display(v.res_modelo.describe().round(3).to_frame("res_modelo"))
print("Fração de resíduos |r|>=8 refeições: modelo", f"{(v.res_modelo.abs()>=8).mean():.2%}", "| D-7", f"{(v.res_d7.abs()>=8).mean():.2%}")
print("Observações mais extremas do modelo:")
display(v.loc[v.res_modelo.abs().sort_values(ascending=False).index[:10], ["data", "curso", "tipo_refeicao", "quantidade_servida", "prev_modelo", "prev_operacional", "res_modelo"]].round({"prev_modelo": 2, "res_modelo": 2}))
print("Viés médio mensal (real − previsto):"); display(mensal.round(2))
print("MAE do modelo por dia da semana:"); display(por_dia.round(3))

Resumo dos resíduos do modelo (2025):


,res_modelo
count,5634.000
mean,0.194
std,2.230
min,-8.834
25%,-1.198
50%,0.000
75%,1.533
max,10.127


Fração de resíduos |r|>=8 refeições: modelo 0.28% | D-7 2.41%
Observações mais extremas do modelo:


,data,curso,tipo_refeicao,quantidade_servida,prev_modelo,prev_operacional,res_modelo
3345,2025-08-05,CURSO_TECNICO_F,almoco,18,7.87,3.0,10.13
5397,2025-12-16,CURSO_TECNICO_F,almoco,19,9.29,12.0,9.71
4247,2025-10-02,CURSO_TECNICO_F,jantar,14,4.69,1.0,9.31
4440,2025-10-15,CURSO_TECNICO_E,almoco,0,8.83,10.0,-8.83
5578,2025-12-27,CURSO_TECNICO_F,cafe_da_manha,10,1.17,1.0,8.83
3480,2025-08-14,CURSO_TECNICO_C,almoco,16,7.41,10.0,8.59
3435,2025-08-11,CURSO_TECNICO_F,almoco,18,9.44,10.0,8.56
5493,2025-12-23,CURSO_TECNICO_B,almoco,1,9.48,12.0,-8.48
1614,2025-04-15,CURSO_TECNICO_E,almoco,0,8.48,11.0,-8.48
5160,2025-12-01,CURSO_TECNICO_E,almoco,15,6.61,8.0,8.39


Viés médio mensal (real − previsto):


,res_trivial,res_d7,res_modelo
m,,,
2025-01-01,-3.27,-0.06,-0.31
2025-02-01,0.58,0.98,-0.04
2025-03-01,0.66,0.11,0.08
2025-04-01,0.62,-0.09,-0.05
2025-05-01,0.73,0.01,0.04
2025-06-01,0.89,-0.09,0.07
2025-07-01,-2.96,-0.79,-0.14
2025-08-01,1.49,1.09,0.69
2025-09-01,1.83,0.09,0.67


MAE do modelo por dia da semana:


,D-7,Modelo Ridge
dia,,
seg,2.337,1.780
ter,2.387,1.829
qua,2.290,1.825
qui,2.272,1.785
sex,2.321,1.776
sáb,1.226,1.219


## 9. Interpretação, limites e declaração de uso de IA

### Quadro obrigatório de comparação (mesma partição, mesmos casos, mesma métrica)

**Validação 2025 — 5.634 linhas `dia × refeição × curso` (dias abertos)**

| Abordagem | Métrica principal: MAE | Métricas secundárias | Observação |
|---|---:|---|---|
| Baseline trivial | 2,67 | RMSE 3,24 · viés +0,28 · MAE total diário 42,91 | média 2019–2024 por curso×refeição |
| Baseline operacional (D-7) | 2,14 | RMSE 2,95 · viés +0,01 · MAE total diário 15,05 | regra **proposta** pela equipe, não confirmada no RU |
| Modelo Ridge | **1,70** | RMSE 2,24 · viés +0,19 · MAE total diário 14,22 | 7 preditoras (30 atributos), α=100 escolhido por CV temporal |

**Teste reservado 2026 (até 14/11) — 4.914 linhas, aberto uma única vez, modelo congelado**

| Abordagem | MAE | RMSE | Viés | MAE total diário |
|---|---:|---:|---:|---:|
| Baseline trivial | 2,88 | 3,49 | +0,69 | 48,29 |
| Baseline operacional (D-7) | 2,20 | 3,01 | +0,10 | **13,84** |
| Modelo Ridge | **1,77** | **2,32** | +0,45 | 15,38 |

Validação cruzada temporal no treino (5 blocos, mesmos casos): MAE médio de 2,45 (trivial), 2,07 (D-7) e 1,63 (modelo). O MAE de CV do modelo é levemente otimista, pois o α foi escolhido nesses blocos.

### O que os números dizem

- **Por linha (dia × refeição × curso), o modelo supera os dois baselines**, na validação e no teste: o MAE cai cerca de 20% em relação ao D-7 (2,14 → 1,70 em 2025; 2,20 → 1,77 em 2026) e o RMSE cai mais ainda. Na validação, o ganho médio sobre o D-7 é de 0,44 refeição por linha, com IC 95% (bootstrap por dia) de [0,36; 0,52]; o modelo é melhor em 55,8% das linhas e pior em 41,1%. O ganho é distinguível de ruído, mas **não é uniforme**: em quase metade das linhas o D-7 acerta mais.
- **No total diário do RU, o ganho some.** Na validação o modelo é só 5% melhor que o D-7 (14,22 contra 15,05) e **no teste é pior** (15,38 contra 13,84). Se a decisão do RU for tomada sobre o total, o D-7 é, neste mock, tão bom quanto o modelo — resultado a registrar, não a esconder.
- **Explicação provável (hipótese, não verificada além dos números):** o viés do modelo é positivo e cresce no tempo (+0,19 por linha em 2025; +0,45 em 2026), isto é, ele **subestima** a demanda. Com 18 linhas por dia (6 cursos × 3 refeições), +0,45 por linha equivale a cerca de 8 refeições a menos por dia. A tendência é linear e foi aprendida em 2019–2024; o mock cresce acima dela no período recente (no viés mensal de 2025, de agosto a dezembro o modelo erra para baixo entre +0,36 e +0,69). O D-7 tem viés quase nulo porque sempre usa a semana mais recente. Subestimar é o tipo de erro que deixa estudantes sem refeição; sem `B` e `C` não dá para dizer se isso é pior que a sobra.
- O trivial é o pior em todas as medidas, como esperado: ignora o padrão semanal e a sazonalidade.
- Os coeficientes maiores do modelo são de calendário: julho (−2,5) e janeiro (−2,1) reduzem a previsão (recessos do calendário simulado), sábado reduz (−1,9) e almoço aumenta (+1,4), em relação às demais categorias. `media_4sem` (+0,75 por desvio-padrão) pesa mais que `lag_7`. Coeficientes de preditoras colineares não devem ser lidos isoladamente.

### Diagnóstico de resíduos (validação 2025)

- **Distribuição:** média +0,19, desvio 2,23, mínimo −8,8 e máximo +10,1; mediana 0. Resíduos com |r| ≥ 8 refeições são 0,28% no modelo e 2,41% no D-7: o modelo evita os grandes erros do D-7 (que repete um dia anormal da semana anterior).
- **Observações extremas:** as dez maiores são oito de almoço, um de jantar e um de café, metade deles do curso F, em agosto, outubro e dezembro; nenhuma foi removida. Sugerem picos ou vales diários do mock, mas **a causa não foi investigada** (o modelo não usa feriados nem eventos).
- **Padrão no tempo:** viés mensal próximo de zero de fevereiro a junho e positivo de agosto a dezembro (subestimativa); janeiro e julho têm viés negativo pequeno. O D-7 oscila em sentido contrário em meses de transição (fevereiro/agosto positivo, dezembro negativo), porque herda a semana anterior.
- **Por dia da semana e refeição:** o MAE do modelo é de 1,78 a 1,83 de segunda a sexta e 1,22 no sábado (volume menor); por refeição, almoço 2,25, café 1,46 e jantar 1,40, sempre abaixo do D-7. A heterocedasticidade (erro maior onde a média é maior, como no almoço) viola a hipótese de variância constante do Ridge, como já registrado.
- Esta seção apresenta os números para o relatório; a análise de resíduos da equipe está a cargo de Kaio Vitor.

### Limites (em linguagem para o demandante)

- **Dados sintéticos:** nada aqui mede o desempenho real no RU/UFPB; o gerador tem regras semanais e sazonais fixas que favorecem tanto o D-7 quanto o modelo.
- **O modelo prevê a média esperada, não choques:** greve, evento, mudança de cardápio, feriado não previsto no calendário e queda de energia não estão nas preditoras.
- **Não usar para decidir subsídio, elegibilidade ou corte de vagas por estudante.** É uma estimativa agregada de refeições por curso.
- **Sem `k`, `B` e `C`** o MAE não é custo econômico; se faltar refeição custar mais que sobrar, uma perda assimétrica (por exemplo, quantílica) deve substituir o MAE e o viés positivo do modelo passa a pesar mais.
- **Depende de dado fechado na véspera:** `lag_7` e `media_4sem` exigem que a catraca de `D-7` esteja completa; se a extração atrasar, o modelo não roda.
- **Curso novo ou sem histórico:** o codificador aceita a categoria, mas `lag_7` e `media_4sem` ficam ausentes e a previsão deixa de ser confiável.
- **Tendência linear:** o modelo não acompanhou o crescimento recente; antes de qualquer uso, reestimá-lo com frequência ou usar um termo de tendência mais flexível.
- Os cortes de partição seguem os da E04 e dependem de confirmação com o RU.

### Uso de IA generativa

Ferramenta: Claude (Claude Code, modelo Sonnet 5.5), como apoio. Finalidade: estruturar este notebook e escrever o código do pipeline (`ColumnTransformer` + `Ridge`), da validação cruzada temporal, das métricas, do teste de bootstrap e dos gráficos, além da redação das interpretações acima. Os baselines e a base por curso reaproveitam o notebook coletivo da E04 (que declara uso de Codex pela coordenação). Os números aqui foram conferidos contra a execução do notebook; a autora deve revisar e ser capaz de explicar cada bloco.
